# OSP — DOTA training run

Trains the Orbital Scene Preprocessor detector on **real aerial imagery** instead of
the synthetic shapes `data/synth_demo.py` draws.

## Before you run

Under **Settings**:

1. **Accelerator** -> `GPU T4` for preference, `GPU P100` only if that is all
   that is offered. Kaggle's preinstalled torch is a cu128 build and the cu128
   wheels carry no Pascal (`sm_60`) kernels, which is what a P100 is, so on a
   P100 every CUDA op fails. The compatibility cell below detects that and pins
   a cu121 torch, but T4 (`sm_75`) needs no downgrade and starts training
   roughly three minutes sooner. Do NOT pick `T4 x2`:
   `model/train_6ch.py` trains on a single CUDA device with no `DataParallel`/DDP
   wrapping, so a second GPU sits idle and you would burn quota twice as fast for
   no speedup.
2. **Internet** -> `On`. The notebook clones GitHub and downloads DOTA.
3. **Persistence** -> optional, but handy if you want to resume.

## Start it with Save Version, not Run All

`Run All` uses the interactive draft session, and Kaggle disconnects that after
about an hour of no clicking in the tab. A multi-hour run started that way dies
part-way through and takes its outputs with it.

Use **Save Version -> "Save & Run All (Commit)"** instead. That executes the whole
notebook as a background batch job with no idle timer, so you can close the tab and
collect `osp_dota_artifacts.zip` from that version's Output tab when it finishes.

## What the timings are sized against

This run is loader-bound, not compute-bound: deriving six bands from a JPEG costs
more than a yolov8n step does. Measured on a 4-core box the tile pipeline sustains
about 29 tiles/s. The 2026-08-23 run measured the corpus DOTA-v1.0 actually
yields: **11,046 train tiles** and 3,677 val, not the ~20,000 this notebook was
originally sized against. At 345 batches per epoch that is roughly 6.5 min per
epoch, so 8+24 epochs lands near **3.5 hours** rather than 7, plus about four
minutes for the DOTA download and tiling. `MAX_HOURS` stays at 9.0 because it
costs nothing to leave headroom for a slower-than-expected runner.

`MAX_HOURS` is the backstop. If this machine turns out slower than that, training
stops cleanly at the budget and the packaging cell still runs, instead of the
session hitting its own limit mid-epoch and discarding everything.

## What comes out

A single file, `osp_dota_artifacts.zip`, in the version's output. Download it,
unzip into the repo, and the rest of the pipeline (INT8 export, briefs, benchmarks)
runs locally on your Mac.

Set `SMOKE = True` in the config cell to rehearse the whole chain on 40 source
images and 2+2 epochs, about 15 minutes, before spending the real hours.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

SMOKE = False    # True: ~15 min end-to-end check. False: the real run.

REPO   = 'https://github.com/brightyorcerf/orbital-preprocessor'
BRANCH = 'main'

# Training. Phase 1 trains the new stem with the backbone frozen; phase 2
# unfreezes and trains the whole network at a lower rate.
#
# Phase 2 is 24 rather than 40 because this run is loader-bound and has a hard
# session limit to fit inside. The 2026-08-23 run measured what DOTA-v1.0 really
# yields: 11,046 train tiles, 345 batches/epoch — not the ~20k this was first
# sized against. At the ~29 tiles/s the pipeline sustains that is ~6.5 min/epoch,
# so 8+24 is about 3.5 hours, not 7. MAX_HOURS below stays generous on purpose:
# it is the backstop if this runner is slower, and unused headroom costs nothing.
EPOCHS_PHASE1 = 2 if SMOKE else 8
EPOCHS_PHASE2 = 2 if SMOKE else 24
BATCH         = 16 if SMOKE else 32

# Hard wall-clock budget for the training stage. When the next epoch would not
# fit, training stops cleanly and the packaging cell still runs. Without this a
# session that hits its own limit mid-epoch is killed outright and the whole
# run produces nothing at all.
MAX_HOURS = 0.25 if SMOKE else 9.0

# Dataloader workers. This is not a minor knob on this run. Tiles are stored as
# RGB JPEG and the six bands are derived on read, which costs tens of
# milliseconds of OpenCV per tile against roughly 10 ms for a yolov8n step, so
# the loader, not the GPU, sets the pace. A Kaggle GPU session has 4 vCPUs, so
# 4 is the number.
WORKERS       = 4

# Tiles used for the per-epoch validation that selects the best checkpoint.
# Validation batches its forward pass and reads through the same worker pool,
# so a useful number is affordable. The full val split is scored once at the
# end regardless.
VAL_LIMIT = 32 if SMOKE else 600
VAL_BATCH = 16

# Tiling. --limit caps source images per split; None means all of them.
LIMIT = 40 if SMOKE else None

print(f'SMOKE={SMOKE}  phase1={EPOCHS_PHASE1}  phase2={EPOCHS_PHASE2}  '
      f'batch={BATCH}  workers={WORKERS}  val_limit={VAL_LIMIT}  limit={LIMIT}  '
      f'max_hours={MAX_HOURS}')
if not SMOKE:
    print('\nREAL RUN. Start it with Save Version -> "Save & Run All (Commit)", '
          'not with Run All:\na committed run executes in the background with no '
          'idle timer to disconnect it.')


## 0. GPU compatibility gate

**This cell is why the 2026-08-23 run failed.** Kaggle's preinstalled torch is a
`cu128` build, and the cu128 wheels dropped the Pascal `sm_60` kernels. Kaggle
still hands out Tesla P100s, which are exactly `sm_60`. The result is a GPU that
imports fine, reports `torch.cuda.is_available() == True`, and then raises
`no kernel image is available for execution on the device` on the first real
CUDA op — which happened 183 seconds in, after the whole 2 GB DOTA download and
tiling had already been paid for.

Note `sm_60` and `sm_61` are not the same thing. Some cu126 builds keep `sm_61`
(GTX 1080 Ti) while still dropping `sm_60` (P100), so "Pascal is supported" is
not a safe reading. `cu121` is the last series that reliably carries `sm_60`.

This must run before anything in this process does `import torch`: pip replacing
torch has no effect on a torch that is already imported.

In [ ]:
# Runs BEFORE any `import torch` in this kernel, deliberately. Detection goes
# through nvidia-smi rather than torch.cuda for the same reason.
import subprocess, re

_q = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap',
                     '--format=csv,noheader'], capture_output=True, text=True)
print(_q.stdout.strip() or _q.stderr.strip())

_rows = [r.strip() for r in _q.stdout.strip().splitlines() if r.strip()]
_cap  = _rows[0].split(',')[-1].strip() if _rows else ''
_major = int(_cap.split('.')[0]) if re.match(r'^\d+\.\d+$', _cap) else None

if _major is None:
    print('\nCould not read compute capability — leaving torch alone. '
          'The kernel-execution test in the next cell is the real gate.')
elif _major < 7:
    print(f'\nsm_{_cap.replace(".", "")} device. The preinstalled cu128 torch has no '
          f'kernels for it.\nPinning a cu121 build, which does. This takes a few minutes '
          f'and is\nstill far cheaper than discovering it after the tiling stage.\n')
    !pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
    print('\npinned.')
else:
    print(f'\nsm_{_cap.replace(".", "")} device — newer than sm_70, so the preinstalled '
          f'torch is fine.\nNo downgrade needed.')

## 1. Confirm the GPU actually executes kernels

`torch.cuda.is_available()` is not a compatibility check. It returned `True` on
the P100 that could not run a single kernel, which is precisely how the previous
run got waved through. This cell launches real kernels instead, including the
exact op that raised on 2026-08-23 (`torch.arange(..., device=cuda)`, allocated
inside `v8DetectionLoss.__init__`), and stops the notebook here if they fail.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), (
    'No CUDA device. Set Accelerator to GPU in the right-hand panel and restart.'
)
cap = torch.cuda.get_device_capability()
print('torch', torch.__version__, '| device:', torch.cuda.get_device_name(0))
print(f'device capability : sm_{cap[0]}{cap[1]}')
print('build arch list   :', torch.cuda.get_arch_list())

# The availability flag above says a device is attached, not that this torch has
# kernels compiled for it. Launch some and find out, in about five seconds.
try:
    _ = torch.arange(16, dtype=torch.float, device='cuda')   # the op that raised
    _a = torch.randn(256, 256, device='cuda')
    _mm = (_a @ _a).sum().item()
    _conv = torch.nn.Conv2d(6, 16, 3, stride=2, padding=1).cuda()   # the 6ch stem
    _out = _conv(torch.randn(2, 6, 64, 64, device='cuda')).sum()
    _out.backward()
    torch.cuda.synchronize()
except Exception as e:
    raise SystemExit(
        f'\nCUDA kernels do NOT execute on this device.\n'
        f'  {type(e).__name__}: {e}\n'
        f'  torch {torch.__version__} built for {torch.cuda.get_arch_list()}\n'
        f'  device is sm_{cap[0]}{cap[1]}\n\n'
        f'Fix: set Accelerator to GPU T4 (sm_75), or adjust the pin in the '
        f'compatibility cell above.'
    ) from e

print(f'\nkernels execute: matmul {_mm:.3f}, 6ch conv backward OK')
del _a, _conv, _out
torch.cuda.empty_cache()

## 2. Clone the repository

The notebook deliberately uses the real repo rather than a copy pasted into
this page, so the code that trains here is the code that is committed.


In [ ]:
import os, shutil, pathlib

WORK = pathlib.Path('/kaggle/working')
TEMP = pathlib.Path('/kaggle/temp')      # scratch; not counted against output quota
TEMP.mkdir(parents=True, exist_ok=True)

SRC = WORK / 'orbital-preprocessor'
if SRC.exists():
    shutil.rmtree(SRC)

!git clone --depth 1 --branch {BRANCH} {REPO} {SRC}
os.chdir(SRC)
print('cwd:', os.getcwd())


In [ ]:
# Kaggle already carries torch, numpy and opencv. Ultralytics supplies the
# YOLOv8 model definition that stem_swap.py operates on.
!pip install -q ultralytics==8.4.125 2>&1 | tail -2
import ultralytics; print('ultralytics', ultralytics.__version__)


### Exercise the real training path on random data first

The kernel test above proves generic ops run. This proves *this repository's*
training step runs: the stem swap to 6 channels, `v8DetectionLoss` construction
(where the previous run actually died), an autocast forward, and a scaled
backward through the optimizer.

It costs about thirty seconds and it sits **before** the 2 GB DOTA download and
the tiling stage, so a broken GPU stack surfaces at minute one instead of minute
four. On random tensors, so it needs no data.

In [ ]:
import sys, gc, logging, contextlib, torch
sys.path.insert(0, '.')
from model.train_6ch import load_or_create_model, build_optimizer
from ultralytics.utils.loss import v8DetectionLoss


@contextlib.contextmanager
def _quiet(level=logging.WARNING):
    """Silence INFO chatter for the duration of the block.

    The stem-swap modules log through `logging`, which writes to stderr. Kaggle
    captures a *notebook process's* stderr twice — once as cell output, once
    into the run log — so every INFO line from this cell shows up duplicated
    with a blank line between. The trainer does not have this problem because it
    runs as a subprocess. Surgery narration is not what this cell is testing and
    it is re-emitted by the trainer anyway, so mute it here rather than ship a
    log that reads like the work happened twice. Exceptions are unaffected: a
    real failure still raises with its full traceback.
    """
    root = logging.getLogger()
    prev = root.level
    root.setLevel(level)
    try:
        yield
    finally:
        root.setLevel(prev)


with _quiet():
    _m = load_or_create_model('model/artifacts/yolov8n_6ch.pt', 'yolov8n.pt', 4, 'cuda')
    _c = v8DetectionLoss(_m)      # <- this is the line that raised on 2026-08-23
    _o = build_optimizer(_m, 1e-3)
    _s = torch.amp.GradScaler("cuda", enabled=True)

# Same dict layout yolo_collate produces: flattened targets plus a batch_idx column.
_b = {
    'img':       torch.rand(2, 6, 640, 640),
    'cls':       torch.zeros(3, 1),
    'bboxes':    torch.tensor([[.5, .5, .2, .2], [.3, .3, .1, .1], [.7, .7, .15, .15]]),
    'batch_idx': torch.tensor([0., 0., 1.]),
}

# These four lines are the actual gate: if the GPU cannot run this repository's
# training step, one of them raises and the notebook stops here, which is the point.
with torch.autocast('cuda'):
    _loss, _parts = _c(_m(_b['img'].cuda()), _b)
_s.scale(_loss.sum()).backward()
_s.step(_o); _s.update(); _o.zero_grad()
torch.cuda.synchronize()

print(f'6-channel forward + backward OK — loss {_loss.sum().item():.3f}')

# The component breakdown is a nicety and is deliberately not allowed to be fatal.
# `v8DetectionLoss` returns a dict keyed box_loss/cls_loss/dfl_loss (verified against
# the pinned ultralytics; run_epoch consumes it as `parts.values()`). An earlier
# draft of this cell indexed it positionally, raised KeyError on the print alone,
# and killed a run three minutes in *after* every CUDA op above had succeeded.
try:
    _named = (dict(_parts) if hasattr(_parts, 'items')
              else dict(zip(('box_loss', 'cls_loss', 'dfl_loss'), _parts)))
    print('  ' + '  '.join(f'{k} {float(v):.3f}' for k, v in _named.items()))
except Exception as _e:
    print(f'  (component breakdown unavailable: {type(_e).__name__}: {_e})')

# Hand the card back: the train cell launches the trainer as a separate process
# that needs the whole GPU, and anything still held here is memory it cannot have.
del _m, _c, _o, _s, _b, _loss, _parts
gc.collect(); torch.cuda.empty_cache()
print(f'GPU after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

## 3. Download DOTA-v1.0

About 2 GB, from the Ultralytics asset mirror. It unpacks to:

```
DOTAv1/
  images/{train,val}/*.png
  labels/{train,val}/*.txt              normalised oriented quads
  labels/{train,val}_original/*.txt     original DOTA annotations
```

`data/dota_prep.py` reads either label format. It uses `labels/{split}/`, which
is the normalised one.

Everything lands in `/kaggle/temp` so it does not consume the 20 GB output quota.


In [ ]:
DOTA_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/DOTAv1.zip'
DOTA_ZIP = TEMP / 'DOTAv1.zip'
DOTA_DIR = TEMP / 'dota'

if not (DOTA_DIR / 'DOTAv1' / 'images').exists():
    if not DOTA_ZIP.exists():
        !curl -fL --retry 3 -o {DOTA_ZIP} {DOTA_URL}
    DOTA_DIR.mkdir(parents=True, exist_ok=True)
    !unzip -q -o {DOTA_ZIP} -d {DOTA_DIR}
    DOTA_ZIP.unlink(missing_ok=True)   # reclaim 2 GB immediately

DOTA_ROOT = DOTA_DIR / 'DOTAv1'

# Fail here, loudly, rather than three cells later inside the tiler. `!curl`
# and `!unzip` do not raise on failure, so an interrupted download otherwise
# surfaces as a confusing "no images found" much further down. -f above also
# stops curl writing an HTML error page into the .zip.
n_train = len(list((DOTA_ROOT / 'images' / 'train').glob('*'))) if (DOTA_ROOT / 'images' / 'train').exists() else 0
n_val   = len(list((DOTA_ROOT / 'images' / 'val').glob('*')))   if (DOTA_ROOT / 'images' / 'val').exists()   else 0
assert n_train > 100 and n_val > 50, (
    f'DOTA did not unpack properly: {n_train} train / {n_val} val images under '
    f'{DOTA_ROOT}. Check internet is enabled and re-run this cell.'
)
print(f'DOTA ready: {n_train} train images, {n_val} val images')


## 4. Tile DOTA into OSP training tiles

This step does four things, each of which is a place the run could go quietly wrong,
so `data/dota_prep.py` reports on all of them:

- keeps only `ship`, `plane`, `storage tank`, `harbor`, discarding DOTA's other eleven categories
- flattens DOTA's **oriented** quads to axis-aligned boxes, and reports the mean area
  inflation this costs (a diagonal ship roughly doubles in box area)
- slices large scenes into 640x640 tiles with overlap, dropping any box that loses
  more than 65% of its area to the crop, so tiling cannot manufacture labels from slivers
- writes **RGB JPEGs**, not 6-band arrays. The six bands are derived at read time.
  A materialised 6-band float32 tile is 9.8 MB; this corpus would be ~200 GB that way,
  against roughly 2 GB as JPEG.


In [ ]:
LIMIT_ARG = f'--limit {LIMIT}' if LIMIT else ''
TILES = TEMP / 'osp_dota'

!python data/dota_prep.py --src {DOTA_ROOT} --out {TILES} {LIMIT_ARG}


In [ ]:
import json
manifest = json.loads((TILES / 'prep_manifest.json').read_text())
for split, s in manifest['splits'].items():
    print(f"{split:6s} {s['tiles_written']:6d} tiles  {s['instances']:7d} instances  "
          f"AABB inflation mean {s['aabb_mean_inflation']} p95 {s['aabb_p95_inflation']}")
    print('       ', s['per_class'])


### Look at a tile before training on 11,000 of them

Boxes drawn from the written label files, not from memory. If the tiling had a
coordinate bug, it is visible here and nowhere else until accuracy comes out wrong.


In [ ]:
# A sanity picture, not a gate. This cell sits immediately before hours of
# training, and in a committed run any exception aborts the whole notebook — so
# a missing thumbnail must not be what stops the GPU work from happening.
try:
    import cv2, random, matplotlib.pyplot as plt
    from pathlib import Path

    NAMES = manifest['classes']
    lbls = [p for p in sorted((TILES/'labels'/'train').glob('*.txt')) if p.read_text().strip()]
    random.seed(0)
    picks = random.sample(lbls, min(3, len(lbls)))
    if not picks:
        print('no labelled training tiles to preview')
    else:
        fig, axes = plt.subplots(1, len(picks), figsize=(5*len(picks), 5), squeeze=False)
        for ax, lf in zip(axes[0], picks):
            raw = cv2.imread(str(TILES/'images'/'train'/(lf.stem+'.jpg')))
            if raw is None:
                ax.set_title(f'{lf.stem}: image missing', fontsize=8); ax.axis('off'); continue
            img = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
            H, W = img.shape[:2]
            for row in lf.read_text().splitlines():
                c, cx, cy, bw, bh = row.split()
                cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
                p1 = (int(cx-bw/2), int(cy-bh/2)); p2 = (int(cx+bw/2), int(cy+bh/2))
                cv2.rectangle(img, p1, p2, (255, 60, 60), 2)
                cv2.putText(img, NAMES[int(c)], (p1[0], max(12, p1[1]-4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 60, 60), 1)
            ax.imshow(img); ax.set_title(lf.stem, fontsize=8); ax.axis('off')
        plt.tight_layout(); plt.show()
except Exception as e:
    print(f'preview skipped ({type(e).__name__}: {e}) — training continues')


## 4a. Is the GPU going to be fed?

This run is loader-bound, not compute-bound: deriving six bands from a JPEG costs
far more than a yolov8n step on a T4. The cell below times the real dataloader
against a real training step, so you find out now rather than five hours in.

If the loader is slower than the step, the GPU idles for the difference and the
whole run stretches by that ratio. Raise `WORKERS`, or cut `EPOCHS_PHASE2`, before
starting.


In [ ]:
# Pre-flight only. This cell allocates a model and a batch on the GPU *in the
# notebook process*, while cell "5. Train" launches the trainer as a separate
# process that needs the whole card to itself. Anything still held here is
# memory that process cannot have, so the measurement runs on the smoke pass —
# where you are choosing epoch counts — and is skipped on the real run.

if not SMOKE:
    print('skipped on the real run (frees the GPU for the training subprocess)')
else:
    import sys, gc, time, torch
    sys.path.insert(0, '.')
    from ground.dataset_6ch import MultiSpectralDataset
    from model.train_6ch import (AugmentedTiles, yolo_collate, worker_init,
                                 load_or_create_model, build_optimizer)
    from ultralytics.utils.loss import v8DetectionLoss

    _ds = AugmentedTiles(MultiSpectralDataset(TILES/'images'/'train',
                                              TILES/'labels'/'train', 640), seed=0)
    _dl = torch.utils.data.DataLoader(
        _ds, batch_size=BATCH, shuffle=True, num_workers=WORKERS,
        collate_fn=yolo_collate, worker_init_fn=worker_init,
        persistent_workers=True, pin_memory=True, prefetch_factor=2, drop_last=True)

    _it = iter(_dl); next(_it)                       # pay worker startup once
    t = time.time(); N = 8
    for _ in range(N): _b = next(_it)
    load_s = (time.time() - t) / N

    _m = load_or_create_model('model/artifacts/yolov8n_6ch.pt', 'yolov8n.pt', 4, 'cuda')
    _c = v8DetectionLoss(_m); _o = build_optimizer(_m, 1e-3); _s = torch.amp.GradScaler("cuda", enabled=True)
    _x = _b['img'].cuda()

    def _step():
        with torch.autocast('cuda'):
            _l, _ = _c(_m(_x), _b)
        _s.scale(_l.sum()).backward(); _s.step(_o); _s.update(); _o.zero_grad()

    for _ in range(3): _step()                       # warm up cudnn autotune
    torch.cuda.synchronize(); t = time.time()
    for _ in range(N): _step()
    torch.cuda.synchronize(); step_s = (time.time() - t) / N

    epochs  = EPOCHS_PHASE1 + EPOCHS_PHASE2
    batches = len(_dl)
    print(f'batch of {BATCH}:  load {load_s*1000:6.0f} ms   gpu step {step_s*1000:6.0f} ms')
    print(f'bound by: {"DATALOADER" if load_s > step_s else "GPU"}   '
          f'utilisation ~{min(load_s, step_s)/max(load_s, step_s)*100:.0f}%')
    print(f'{batches} batches/epoch x {epochs} epochs at the SMOKE settings.')
    print(f'Scale by your real tile count to size EPOCHS_PHASE2 and MAX_HOURS.')

    # Shut the worker pool down explicitly (persistent_workers keeps it alive
    # past `del`), drop every CUDA reference, then hand the memory back.
    _it2 = getattr(_dl, '_iterator', None)
    if _it2 is not None: _it2._shutdown_workers()
    del _it, _it2, _dl, _ds, _m, _c, _o, _s, _x, _b, _step
    gc.collect(); torch.cuda.empty_cache()
    print(f'\nGPU after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')


## 5. Train

`model/train_6ch.py` performs the stem swap (3 channels/80 classes -> 6 channels/4 classes)
and then runs its own two-phase loop. It does not call `YOLO().train()`, because
Ultralytics' data pipeline cannot read 6-channel float tiles.

Expect the number to land far below the synthetic model's 0.99. **That is the point
of this run.** A moderate score on real aerial imagery means something; a perfect
score on shapes the repository drew itself does not.


In [ ]:
import time
t0 = time.time()

# Mixed precision and the weight EMA are on by default on CUDA
# (--no-amp / --ema-decay 0 turn them off). --max-hours stops training cleanly
# rather than letting the session limit kill it mid-epoch.
!python model/train_6ch.py \
    --dataset {TILES} \
    --device cuda \
    --epochs-phase1 {EPOCHS_PHASE1} \
    --epochs {EPOCHS_PHASE2} \
    --batch {BATCH} \
    --workers {WORKERS} \
    --val-limit {VAL_LIMIT} \
    --val-batch {VAL_BATCH} \
    --max-hours {MAX_HOURS} \
    --out model/artifacts/osp_best.pt \
    --metrics-out model/artifacts/train_metrics.json

print(f'\ntraining wall clock: {(time.time()-t0)/60:.1f} min')


In [ ]:
from pathlib import Path
import json

_mf = Path('model/artifacts/train_metrics.json')
if _mf.exists():
    metrics = json.loads(_mf.read_text())
    print(json.dumps({k: v for k, v in metrics.items() if k != 'history'}, indent=2)[:2000])

    # train_6ch.py rewrites this file after every epoch, so its presence no
    # longer means the run finished — only `status` does.
    if metrics.get('status') == 'in_progress':
        print(f"\nPARTIAL: {metrics['epochs_total']} of {metrics['epochs_requested']} "
              f"epochs recorded, then the run stopped without completing its final "
              f"full-val scoring. osp_best.pt is still the best checkpoint of those "
              f"epochs and the packaging cell will ship it; re-score it locally "
              f"against the val split in the zip.")
    elif metrics.get('stopped_early_on_budget'):
        print(f"\nNOTE: stopped at {metrics['epochs_total']} of "
              f"{metrics['epochs_requested']} epochs on the MAX_HOURS budget.")
else:
    # Do not raise: an exception here aborts the commit, and the packaging cell
    # below is the only thing that gets a checkpoint out of this session.
    metrics = None
    print('NO METRICS FILE — training did not reach the end of its first epoch. '
          'The next cell will still package whatever exists.')

## 6. Package the results

Only the checkpoint, the metrics and the tiling manifest come back. The tiles
themselves stay here: they are reproducible from `data/dota_prep.py` and a copy
of DOTA, so shipping gigabytes of them home would be pointless.

The validation tiles **do** come back, because the INT8 calibration and the
accuracy re-scoring you run locally have to use the same held-out split this
model was scored against. Calibrating on a different distribution than the one
the model runs on is the classic way to lose small-object recall silently.


In [ ]:
import shutil, os, json
from pathlib import Path

# Staged in /kaggle/temp, not /kaggle/working. copytree of the val split is
# thousands of files, and /kaggle/working is the quota-counted output directory
# with an item-count limit as well as a size one. Only the finished zip belongs
# there.
OUT = TEMP / 'osp_dota_out'
if OUT.exists(): shutil.rmtree(OUT)
(OUT / 'model' / 'artifacts').mkdir(parents=True, exist_ok=True)

kept, missing = [], []
for f in ['osp_best.pt', 'yolov8n_6ch.pt', 'train_metrics.json']:
    src = Path('model/artifacts') / f
    (kept if src.exists() else missing).append(f)
    if src.exists(): shutil.copy2(src, OUT / 'model' / 'artifacts' / f)

for f in ['prep_manifest.json', 'dataset.yaml']:
    if (TILES / f).exists():
        shutil.copy2(TILES / f, OUT / f)
        kept.append(f)
    else:
        missing.append(f)

# Held-out validation split, for local INT8 calibration and re-scoring. The
# calibration set has to be the split the model was scored against; calibrating
# on a different distribution is how small-object recall disappears silently.
for sub in ['images', 'labels']:
    srcd = TILES / sub / 'val'
    if srcd.exists():
        shutil.copytree(srcd, OUT / 'val' / sub)
        kept.append(f'val/{sub} ({len(list(srcd.glob("*")))} files)')

archive = shutil.make_archive(str(WORK / 'osp_dota_artifacts'), 'zip', str(OUT))
shutil.rmtree(OUT)

print('packaged:')
for k in kept: print('  +', k)
if missing:
    print('MISSING (training may not have completed):')
    for k in missing: print('  -', k)
print(f'\nwrote {archive} ({os.path.getsize(archive)/1e6:.1f} MB)')
print('Download osp_dota_artifacts.zip from the Output panel of this version.')
